# Imports and Setup
Import necessary libraries for data manipulation, modeling, and visualization.

In [40]:
import pandas as pd
import statsmodels.api as sm
import numpy as np
import shap
import warnings
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.exceptions import ConvergenceWarning
pd.set_option('display.max_columns', None)

#Filter convergence warning
warnings.filterwarnings("ignore", category=ConvergenceWarning)

# Feature Selection
Define the list of columns to be used for the analysis, focusing on loan characteristics and borrower information.

In [41]:
cols_to_keep = [
    'loan_amnt', 'int_rate','term', 'home_ownership', 
    'installment', 'sub_grade', 'emp_length', 'annual_inc', 'loan_status', 
    'purpose', 'fico_range_low', 'delinq_2yrs'
]

# Data Loading
Load the LendingClub dataset from the compressed CSV file.

In [42]:
raw_data = pd.read_csv('accepted_2007_to_2018Q4.csv.gz', compression='gzip', low_memory=False)

# Data Inspection
Display the first few rows of the dataset to understand its structure and content.

In [43]:
raw_data.head()

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,url,desc,purpose,title,zip_code,addr_state,dti,delinq_2yrs,earliest_cr_line,fico_range_low,fico_range_high,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,next_pymnt_d,last_credit_pull_d,last_fico_range_high,last_fico_range_low,collections_12_mths_ex_med,mths_since_last_major_derog,policy_code,application_type,annual_inc_joint,dti_joint,verification_status_joint,acc_now_delinq,tot_coll_amt,tot_cur_bal,open_acc_6m,open_act_il,open_il_12m,open_il_24m,mths_since_rcnt_il,total_bal_il,il_util,open_rv_12m,open_rv_24m,max_bal_bc,all_util,total_rev_hi_lim,inq_fi,total_cu_tl,inq_last_12m,acc_open_past_24mths,avg_cur_bal,bc_open_to_buy,bc_util,chargeoff_within_12_mths,delinq_amnt,mo_sin_old_il_acct,mo_sin_old_rev_tl_op,mo_sin_rcnt_rev_tl_op,mo_sin_rcnt_tl,mort_acc,mths_since_recent_bc,mths_since_recent_bc_dlq,mths_since_recent_inq,mths_since_recent_revol_delinq,num_accts_ever_120_pd,num_actv_bc_tl,num_actv_rev_tl,num_bc_sats,num_bc_tl,num_il_tl,num_op_rev_tl,num_rev_accts,num_rev_tl_bal_gt_0,num_sats,num_tl_120dpd_2m,num_tl_30dpd,num_tl_90g_dpd_24m,num_tl_op_past_12m,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,revol_bal_joint,sec_app_fico_range_low,sec_app_fico_range_high,sec_app_earliest_cr_line,sec_app_inq_last_6mths,sec_app_mort_acc,sec_app_open_acc,sec_app_revol_util,sec_app_open_act_il,sec_app_num_rev_accts,sec_app_chargeoff_within_12_mths,sec_app_collections_12_mths_ex_med,sec_app_mths_since_last_major_derog,hardship_flag,hardship_type,hardship_reason,hardship_status,deferral_term,hardship_amount,hardship_start_date,hardship_end_date,payment_plan_start_date,hardship_length,hardship_dpd,hardship_loan_status,orig_projected_additional_accrued_interest,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,NaN,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,leadman,10+ years,MORTGAGE,55000.0,Not Verified,Dec-2015,Fully Paid,n,https://lendingclub.com/browse/loanDetail.acti...,NaN,debt_consolidation,Debt consolidation,190xx,PA,5.91,0.0,Aug-2003,675.0,679.0,1.0,30.0,NaN,7.0,0.0,2765.0,29.7,13.0,w,0.00,0.00,4421.723917,4421.72,3600.00,821.72,0.0,0.0,0.0,Jan-2019,122.67,NaN,Mar-2019,564.0,560.0,0.0,30.0,1.0,Individual,NaN,NaN,NaN,0.0,722.0,144904.0,2.0,2.0,0.0,1.0,21.0,4981.0,36.0,3.0,3.0,722.0,34.0,9300.0,3.0,1.0,4.0,4.0,20701.0,1506.0,37.2,0.0,0.0,148.0,128.0,3.0,3.0,1.0,4.0,69.0,4.0,69.0,2.0,2.0,4.0,2.0,5.0,3.0,4.0,9.0,4.0,7.0,0.0,0.0,0.0,3.0,76.9,0.0,0.0,0.0,178050.0,7746.0,2400.0,13734.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,68355089,NaN,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,Engineer,10+ years,MORTGAGE,65000.0,Not Verified,Dec-2015,Fully Paid,n,https://lendingclub.com/browse/loanDetail.acti...,NaN,small_business,Business,577xx,SD,16.06,1.0,Dec-1999,715.0,719.0,4.0,6.0,NaN,22.0,0.0,21470.0,19.2,38.0,w,0.00,0.00,25679.660000,25679.66,24700.00,979.66,0.0,0.0,0.0,Jun-2016,926.35,NaN,Mar-2019,699.0,695.0,0.0,NaN,1.0,Individual,NaN,NaN,NaN,0.0,0.0,204396.0,1.0,1.0,0.0,1.0,19.0,18005.0,73.0,2.0,3.0,6472.0,29.0,111800.0,0.0,0.0,6.0,4.0,9733.0,57830.0,27.1,0.0,0.0,113.0,192.0,2.0,2.0,4.0,2.0,NaN,0.0,6.0,0.0,5.0,5.0,13.0,17.0,6.0,20.0,27.0,5.0,22.0,0.0,0.0,0.0,2.0,97.4,7.7,0.0,0.0,314017.0,39475.0,79300.0,24667.0,NaN,NaN,NaN,NaN,NaN,N

# Filtering & Target Creation
Filter the dataset to keep only fully paid, charged off, and defaulted loans. Create a binary target variable where 1 represents default/charged off and 0 represents fully paid.

In [44]:
data = raw_data[cols_to_keep].copy()

In [45]:
data.head()

,loan_amnt,int_rate,term,home_ownership,installment,sub_grade,emp_length,annual_inc,loan_status,purpose,fico_range_low,delinq_2yrs
0,3600.0,13.99,36 months,MORTGAGE,123.03,C4,10+ years,55000.0,Fully Paid,debt_consolidation,675.0,0.0
1,24700.0,11.99,36 months,MORTGAGE,820.28,C1,10+ years,65000.0,Fully Paid,small_business,715.0,1.0
2,20000.0,10.78,60 months,MORTGAGE,432.66,B4,10+ years,63000.0,Fully Paid,home_improvement,695.0,0.0
3,35000.0,14.85,60 months,MORTGAGE,829.90,C5,10+ years,110000.0,Current,debt_consolidation,785.0,0.0
4,10400.0,22.45,60 months,MORTGAGE,289.91,F1,3 years,104433.0,Fully Paid,major_purchase,695.0,1.0


In [46]:
data = data.loc[data['loan_status'].isin(['Fully Paid', 'Charged Off', 'Default'])]
data['target'] = data['loan_status'].apply(lambda x: 1 if x in ['Charged Off', 'Default'] else 0)
data.drop('loan_status', axis=1, inplace=True)

# Feature Engineering (Term)
Convert the 'term' column from a string (e.g., ' 36 months') to a numerical value.

In [47]:
if data['term'].dtype == object:
    data['term'] = data['term'].str.extract('(\d+)').astype(int)

# Feature Engineering (Employment)
Map employment length strings to numerical values for modeling.

In [48]:
emp_map = {
    '10+ years': 10,
    '9 years': 9,
    '8 years': 8,
    '7 years': 7,
    '6 years': 6,
    '5 years': 5,
    '4 years': 4,
    '3 years': 3,
    '2 years': 2,
    '1 year': 1,
    '< 1 year': 0
}

if data['emp_length'].dtype == object:
    data['emp_length'] = data['emp_length'].map(emp_map)

# Class Imbalance Check
Calculate the proportion of defaults in the dataset to check for class imbalance.

In [49]:
data['target'].mean()

0.19964990522912254

# Train-Test Split
Split the data into training and testing sets to evaluate model performance on unseen data. Stratified split is used to maintain class distribution.

In [50]:
X = data.drop('target', axis=1)
y = data['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=123456, stratify=y)
X_train = X_train.drop('int_rate', axis=1)

# Preprocessing Pipeline
Define preprocessing steps for numeric (imputation, scaling) and categorical (imputation, one-hot encoding) features.

In [51]:
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
numeric_features = numeric_features.drop('int_rate')
categorical_features = X.select_dtypes(include=['object', 'category']).columns

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Logistic Regression Model
Initialize a Logistic Regression model within a pipeline that includes the preprocessing steps. Class weights are balanced to handle the imbalanced dataset.

In [52]:
logmodel = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=123456, class_weight='balanced', solver='saga', max_iter=100))  
])

In [53]:
logmodel.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


# Model Evaluation
Evaluate the model's performance on the test set using precision, recall, F1-score, and ROC-AUC score.

After checking AOC-ROC score and that logistic regression is just as good and even better than random forest while keeping the statistical interpretability I've chosen the logistic regression for profitability report.

In [54]:
y_pred = logmodel.predict(X_test)
y_prob = logmodel.predict_proba(X_test)[:, 1]

print("Classification Report:\n", classification_report(y_test, y_pred))
print("ROC-AUC Score:", roc_auc_score(y_test, y_prob))

Classification Report:
               precision    recall  f1-score   support

           0       0.88      0.61      0.72    215350
           1       0.30      0.68      0.42     53720

    accuracy                           0.63    269070
   macro avg       0.59      0.65      0.57    269070
weighted avg       0.77      0.63      0.66    269070

ROC-AUC Score: 0.7010587702386166


In [55]:
rf_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=123456, class_weight='balanced', n_estimators=100, max_depth=10))
])

In [56]:
rf_model.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


# Feature importance in Random Forest model

Evaluate the feature importances using mean decrease in impurity

In [57]:
importances = rf_model.named_steps['classifier'].feature_importances_

feature_names = rf_model[:-1].get_feature_names_out()
numerical_cols = X.select_dtypes(include=['number']).columns.tolist()

feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances}).sort_values(by='Importance', ascending=False)

feature_numeric_importance_df =feature_importance_df[feature_importance_df['Feature'].str.startswith('num__')]

feature_numeric_importance_df

,Feature,Importance
4,num__fico_range_low,0.205223
0,num__loan_amnt,0.081003
3,num__annual_inc,0.062245
1,num__installment,0.060389
2,num__emp_length,0.007255
5,num__delinq_2yrs,0.003005


In [58]:
y_pred_rf = rf_model.predict(X_test)
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_rf))
print("ROC-AUC Score:", roc_auc_score(y_test, y_prob_rf))

              precision    recall  f1-score   support

           0       0.88      0.57      0.69    215350
           1       0.29      0.69      0.41     53720

    accuracy                           0.60    269070
   macro avg       0.58      0.63      0.55    269070
weighted avg       0.76      0.60      0.64    269070

ROC-AUC Score: 0.6816477115817452


In [59]:
preprocessor.set_params(num__scaler=None)
preprocessor.fit(X_train)

X_train_processed = preprocessor.transform(X_train)

cat_names = preprocessor.named_transformers_['cat']['onehot'].get_feature_names_out(categorical_features)
all_feature_names = numeric_features.tolist() + cat_names.tolist()

X_train_processed = X_train_processed.toarray()

X_train_sm_df = pd.DataFrame(X_train_processed, columns=all_feature_names)
X_train_sm_df = sm.add_constant(X_train_sm_df)


In [60]:
logit_model = sm.Logit(y_train.reset_index(drop=True), X_train_sm_df)
result = logit_model.fit()

Optimization terminated successfully.
         Current function value: 0.458140
         Iterations 7


In [61]:
result.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:                 target   No. Observations:              1076280
Model:                          Logit   Df Residuals:                  1076221
Method:                           MLE   Df Model:                           58
Date:                Wed, 11 Mar 2026   Pseudo R-squ.:                 0.08357
Time:                        22:52:50   Log-Likelihood:            -4.9309e+05
converged:                       True   LL-Null:                   -5.3805e+05
Covariance Type:            nonrobust   LLR p-value:                     0.000
==============================================================================================
                                 coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------
const                         -0.3544      0.194     -1.824      0.068      -0.735       0.027
loan_amnt                    6.48e-05   9.16e-07     70.734      0.000     6.3e-05    6.66e-05
installment                   -0.0016   2.99e-05    -52.088      0.000      -0.002      -0.001
emp_length                    -0.0041      0.001     -5.752      0.000      -0.006      -0.003
annual_inc                 -2.965e-06   7.03e-08    -42.174      0.000    -3.1e-06   -2.83e-06
fico_range_low                -0.0040      0.000    -37.021      0.000      -0.004      -0.004
delinq_2yrs                    0.0205      0.003      7.454      0.000       0.015       0.026
home_ownership_MORTGAGE       -0.2177      0.171     -1.271      0.204      -0.553       0.118
home_ownership_NONE           -0.4291      0.464     -0.926      0.355      -1.338       0.480
home_ownership_OTHER           0.1473      0.299      0.492      0.623      -0.439       0.734
home_ownership_OWN            -0.0171      0.171     -0.099      0.921      -0.353       0.319
home_ownership_RENT            0.1044      0.171      0.610      0.542      -0.231       0.440
sub_grade_A2                   0.2642      0.041      6.455      0.000       0.184       0.344
sub_grade_A3                   0.4242      0.039     10.827      0.000       0.347       0.501
sub_grade_A4                   0.5902      0.036     16.442      0.000       0.520       0.661
sub_grade_A5                   0.7631      0.034     22.272      0.000       0.696       0.830
sub_grade_B1                   0.9778      0.033     29.317      0.000       0.912       1.043
sub_grade_B2                   1.0421      0.033     31.470      0.000       0.977       1.107
sub_grade_B3                   1.1698      0.033     35.797      0.000       1.106       1.234
sub_grade_B4                   1.3204      0.032     40.667      0.000       1.257       1.384
sub_grade_B5                   1.4405      0.032     44.464      0.000       1.377       1.504
sub_grade_C1                   1.5757      0.032     48.900      0.000       1.513       1.639
sub_grade_C2                   1.6603      0.032     51.427      0.000       1.597       1.724
sub_grade_C3                   1.7620      0.032     54.548      0.000       1.699       1.825
sub_grade_C4                   1.8744      0.032     58.099      0.000       1.811       1.938
sub_grade_C5                   1.9286      0.032     59.555      0.000       1.865       1.992
sub_grade_D1                   2.0198      0.033     61.535      0.000       1.955       2.084
sub_grade_D2                   2.1067      0.033     63.758      0.000       2.042       2.171
sub_grade_D3                   2.1484      0.033     64.529      0.000       2.083       2.214
sub_grade_D4                   2.2081      0.033     65.919      0.000       2.142       2.274
sub_grade_D5                   2.2524      0.034     66.392      0.000       2.186       2.319
sub_grade_E

In [62]:
results = pd.DataFrame({
    'loan_amnt': X_test['loan_amnt'],
    'actual_default': y_test,
    'prob_default': y_prob,
    'int_rate': X_test['int_rate'],
    'term': X_test['term']
})
results.head()

,loan_amnt,actual_default,prob_default,int_rate,term
1300171,24000.0,0,0.292630,6.03,36
1031884,15075.0,0,0.470733,13.67,36
1812062,10925.0,0,0.582090,17.56,60
1630807,5625.0,0,0.511985,13.43,60
1306164,15000.0,1,0.645576,18.25,60


# Business Analysis: Profit Calculation
Define a function to calculate the profit or loss for each loan based on its interest rate, term, and default status.

In [63]:
def calculate_profit(row, threshold):
    years = row['term'] / 12
    if row['prob_default'] > threshold:
        return 0
    
    if row['actual_default'] == 1:
        return -row['loan_amnt'] 
    else:
        return row['loan_amnt'] * (row['int_rate'] / 100) * years 

In [64]:
thresholds = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]

In [65]:
for t in thresholds:
    results[f'profit_{t}'] = results.apply(calculate_profit, threshold=t, axis=1)
    print(f"Threshold {t}: Total Profit = ${results[f'profit_{t}'].sum():,.0f}")

Threshold 0.2: Total Profit = $38,144,279
Threshold 0.3: Total Profit = $117,046,312
Threshold 0.4: Total Profit = $218,032,468
Threshold 0.5: Total Profit = $329,066,409
Threshold 0.6: Total Profit = $452,523,156
Threshold 0.7: Total Profit = $564,795,433


In [66]:
results_rf = pd.DataFrame({
    'loan_amnt': X_test['loan_amnt'],
    'actual_default': y_test,
    'prob_default': y_prob_rf,
    'int_rate': X_test['int_rate'],
    'term': X_test['term']
})
results_rf.head()

,loan_amnt,actual_default,prob_default,int_rate,term
1300171,24000.0,0,0.474634,6.03,36
1031884,15075.0,0,0.571542,13.67,36
1812062,10925.0,0,0.570877,17.56,60
1630807,5625.0,0,0.433615,13.43,60
1306164,15000.0,1,0.590583,18.25,60


In [67]:
for t in thresholds:
    results_rf[f'profit_{t}'] = results_rf.apply(calculate_profit, threshold=t, axis=1)
    print(f"Threshold {t}: Total Profit = ${results_rf[f'profit_{t}'].sum():,.0f}")

Threshold 0.2: Total Profit = $2,475,055
Threshold 0.3: Total Profit = $39,532,708
Threshold 0.4: Total Profit = $139,508,071
Threshold 0.5: Total Profit = $331,575,224
Threshold 0.6: Total Profit = $591,810,248
Threshold 0.7: Total Profit = $635,652,913


# Stress Testing
Simulate different economic scenarios (e.g., increased default rates) to assess the portfolio's resilience and profitability under stress.

In [68]:
def run_stress_test(df, stress_factor=1.0):
    stressed_df = df.copy()
    
    safe_loans_indices = stressed_df[stressed_df['actual_default'] == 0].index
    
    current_default_rate = stressed_df['actual_default'].mean()
    target_default_rate = current_default_rate * stress_factor
    
    if target_default_rate > 1.0: target_default_rate = 1.0
    
    total_loans = len(stressed_df)
    n_current_defaults = stressed_df['actual_default'].sum()
    n_target_defaults = int(total_loans * target_default_rate)
    n_to_flip = n_target_defaults - n_current_defaults
    
    if n_to_flip > 0:
        flip_indices = np.random.choice(safe_loans_indices, size=n_to_flip, replace=False)
        stressed_df.loc[flip_indices, 'actual_default'] = 1
        
    return stressed_df

In [69]:
scenarios = {
    "Baseline (Normal)": 1.0,
    "Mild Recession (+50% Defaults)": 1.5,
    "2008 Crisis (+100% Defaults)": 2.0
}

In [70]:
for t in thresholds:
    for name, factor in scenarios.items():
        df_stressed = run_stress_test(results_rf, stress_factor=factor)
        
        df_stressed['profit'] = df_stressed.apply(calculate_profit, threshold=t, axis=1)
        
        total_profit = df_stressed['profit'].sum()
        print(f"Threshold {t} | {name:<30} | Profit: ${total_profit:,.0f}")

Threshold 0.2 | Baseline (Normal)              | Profit: $2,475,055
Threshold 0.2 | Mild Recession (+50% Defaults) | Profit: $395,647
Threshold 0.2 | 2008 Crisis (+100% Defaults)   | Profit: $-2,109,855
Threshold 0.3 | Baseline (Normal)              | Profit: $39,532,708
Threshold 0.3 | Mild Recession (+50% Defaults) | Profit: $6,714,925
Threshold 0.3 | 2008 Crisis (+100% Defaults)   | Profit: $-30,032,570
Threshold 0.4 | Baseline (Normal)              | Profit: $139,508,071
Threshold 0.4 | Mild Recession (+50% Defaults) | Profit: $20,510,261
Threshold 0.4 | 2008 Crisis (+100% Defaults)   | Profit: $-97,613,464
Threshold 0.5 | Baseline (Normal)              | Profit: $331,575,224
Threshold 0.5 | Mild Recession (+50% Defaults) | Profit: $67,869,640
Threshold 0.5 | 2008 Crisis (+100% Defaults)   | Profit: $-191,723,894
Threshold 0.6 | Baseline (Normal)              | Profit: $591,810,248
Threshold 0.6 | Mild Recession (+50% Defaults) | Profit: $108,262,604
Threshold 0.6 | 2008 Crisis (+1